# Reconstruction-Probability Scoring — A Genuinely Different Architecture

**Why this is different from the two failed hyperparameter-tuning attempts**:
those only changed *how much capacity/regularization* the same architecture
gets. This changes *what the model outputs and how the anomaly score is
computed*.

**The mechanism (An & Cho, 2015-style)**: the current model's decoder outputs
one number per feature (the reconstruction), and the anomaly score is plain
MSE — every one of the 26 PCA components is treated as equally uncertain.
But `windowing_pca.ipynb`'s own analysis showed the components are wildly
unequal in how much genuine signal they carry (PC1+PC2 = 92.7% of variance,
memory-dominated; the fault-relevant tail components share the remaining
7.3%). Plain MSE has no way to know that.

**The fix**: give the decoder a SECOND output head that predicts its own
*confidence* (a variance) for each of the 26 components, and score using the
resulting Gaussian negative log-likelihood instead of MSE:

```
score = mean_over_dims( (x - reconstruction)^2 * exp(-predicted_logvar) + predicted_logvar )
```

A miss on a component the model has learned to expect is noisy costs less;
a miss on a component it expects to be precise costs more. This is a
different, principled way to address exactly the fragile-tail-signal
diagnosis from the previous investigation — not a parameter retune.

**Full retrain required** — this is a genuine architecture change (the
decoder's output shape changes), so the beta/latent_dim search has to be
redone for this new loss formulation using the same valid two-phase
methodology validated in `hyperparameter_tuning_v2.ipynb` (beta chosen by
collapse-check only, latent_dim compared within one fixed beta).

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle, os, time
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                              recall_score, f1_score, precision_recall_curve)

torch.manual_seed(42)
np.random.seed(42)

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')
OUT_DIR   = os.path.join(BASE, 'experiments')

DEVICE = torch.device('cpu')
ALL_SETS = ['cc1_test', 'drift_cc2']

HIDDEN1, HIDDEN2 = 64, 32
WARMUP_EPOCHS = 10
SCREEN_EPOCHS, SCREEN_PATIENCE = 40, 8
FULL_MAX_EPOCHS, FULL_PATIENCE = 300, 20
LR, BATCH_SIZE, CLIP = 1e-3, 512, 20.0
COLLAPSE_KL_THRESH = 0.05

BETA_SEARCH_LATENT_DIM = 16
BETA_GRID = [0.1, 0.03, 0.01, 0.003, 0.001, 0.0003, 0.0001, 0.00003]
LATENT_GRID = [16, 20, 24, 28, 32, 40]

print(f'Phase A beta grid (fixed latent_dim={BETA_SEARCH_LATENT_DIM}): {BETA_GRID}')
print(f'Phase B latent_dim grid: {LATENT_GRID}')

Phase A beta grid (fixed latent_dim=16): [0.1, 0.03, 0.01, 0.003, 0.001, 0.0003, 0.0001, 3e-05]
Phase B latent_dim grid: [16, 20, 24, 28, 32, 40]


## Step 1 — Load data

In [2]:
raw = {name: np.load(os.path.join(DATA_DIR, f'X_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
labels = {name: np.load(os.path.join(DATA_DIR, f'y_{name}.npy')) for name in ['cc1_train', 'cc1_val'] + ALL_SETS}
data = {name: np.clip(X, -CLIP, CLIP).astype(np.float32) for name, X in raw.items()}
INPUT_DIM = data['cc1_train'].shape[1]

for name, X in raw.items():
    assert not np.isnan(X).any() and not np.isinf(X).any(), f'{name}: NaN/Inf found'
assert (labels['cc1_train'] == 0).all()

X_train_t = torch.from_numpy(data['cc1_train'])
X_val_t   = torch.from_numpy(data['cc1_val'])
print(f'INPUT_DIM={INPUT_DIM}  X_train={X_train_t.shape}  X_val={X_val_t.shape}')
print('Sanity checks passed.')

INPUT_DIM=26  X_train=torch.Size([154198, 26])  X_val=torch.Size([21573, 26])
Sanity checks passed.


## Step 2 — VAE with a two-headed decoder (mean + log-variance) and Gaussian NLL loss

In [3]:
class ReconProbVAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder_trunk = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU())
        self.fc_dec_mu = nn.Linear(hidden1, input_dim)
        self.fc_dec_lv = nn.Linear(hidden1, input_dim)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)

    def decode(self, z):
        h = self.decoder_trunk(z)
        mu_x = self.fc_dec_mu(h)
        logvar_x = torch.clamp(self.fc_dec_lv(h), -10, 10)
        return mu_x, logvar_x

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        mu_x, logvar_x = self.decode(z)
        return mu_x, logvar_x, mu_z, logvar_z

    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval()
        mu_z, _ = self.encode(x)
        mu_x, logvar_x = self.decode(mu_z)
        sq_err = (x - mu_x) ** 2
        return (sq_err * torch.exp(-logvar_x) + logvar_x).mean(dim=1)

def reconprob_loss(mu_x, logvar_x, x, mu_z, logvar_z, beta):
    sq_err = (x - mu_x) ** 2
    recon_nll = (sq_err * torch.exp(-logvar_x) + logvar_x).mean()
    kl = -0.5 * torch.mean(torch.sum(1 + logvar_z - mu_z.pow(2) - logvar_z.exp(), dim=1))
    return recon_nll + beta * kl, recon_nll, kl

def train_reconprob_vae(latent_dim, beta_max, max_epochs, patience, seed=42):
    torch.manual_seed(seed)
    model = ReconProbVAE(INPUT_DIM, HIDDEN1, HIDDEN2, latent_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(X_train_t), batch_size=BATCH_SIZE, shuffle=True)
    best_val, best_state, patience_ctr = float('inf'), None, 0
    final_kl, final_recon = None, None

    for epoch in range(max_epochs):
        beta = min(1.0, (epoch + 1) / WARMUP_EPOCHS) * beta_max
        model.train()
        for (xb,) in loader:
            opt.zero_grad()
            mu_x, logvar_x, mu_z, logvar_z = model(xb)
            loss, rloss, kl = reconprob_loss(mu_x, logvar_x, xb, mu_z, logvar_z, beta)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            mu_x, logvar_x, mu_z, logvar_z = model(X_val_t)
            vloss, vrecon, vkl = reconprob_loss(mu_x, logvar_x, X_val_t, mu_z, logvar_z, beta)
        final_kl, final_recon = vkl.item(), vrecon.item()
        if vloss.item() < best_val - 1e-6:
            best_val, best_state, patience_ctr = vloss.item(), {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, final_kl, final_recon, epoch + 1

print('ReconProbVAE + training function defined.')
n_params = sum(p.numel() for p in ReconProbVAE(INPUT_DIM, HIDDEN1, HIDDEN2, 32).parameters())
print(f'Parameter count at latent_dim=32: {n_params:,} (vs 10,778 for the original single-headed decoder)')

ReconProbVAE + training function defined.
Parameter count at latent_dim=32: 12,468 (vs 10,778 for the original single-headed decoder)


## Phase A — Find the collapse boundary for THIS loss formulation (fixed latent_dim=16)

The NLL loss has a different natural scale than plain MSE (the `logvar_x`
term can shift things substantially), so the beta that worked for the
original model does not necessarily transfer — this has to be re-searched,
not assumed.

In [4]:
beta_search = {}
for beta_max in BETA_GRID:
    t0 = time.time()
    _, best_val, final_kl, final_recon, n_epochs = train_reconprob_vae(BETA_SEARCH_LATENT_DIM, beta_max, SCREEN_EPOCHS, SCREEN_PATIENCE)
    elapsed = time.time() - t0
    collapsed = final_kl < COLLAPSE_KL_THRESH
    beta_search[beta_max] = {'best_val': best_val, 'final_kl': final_kl, 'final_recon': final_recon, 'collapsed': collapsed}
    print(f'  beta={beta_max:.5f}  val_loss={best_val:.4f}  final_kl={final_kl:.4f}  final_recon={final_recon:.4f}  '
          f'{"COLLAPSED" if collapsed else "ok"}  ({elapsed:.0f}s)')

non_collapsed = [b for b, r in beta_search.items() if not r['collapsed']]
assert non_collapsed, 'Every beta candidate collapsed.'
BEST_BETA = max(non_collapsed)
print(f'\nLargest non-collapsing beta_max = {BEST_BETA}')

  beta=0.10000  val_loss=0.9212  final_kl=1.2443  final_recon=0.7968  ok  (86s)
  beta=0.03000  val_loss=0.7703  final_kl=5.6198  final_recon=0.6017  ok  (85s)
  beta=0.01000  val_loss=-0.1514  final_kl=40.6246  final_recon=-0.5577  ok  (87s)
  beta=0.00300  val_loss=-0.6591  final_kl=73.7730  final_recon=-0.8804  ok  (88s)
  beta=0.00100  val_loss=-0.7462  final_kl=112.1021  final_recon=-0.8583  ok  (88s)
  beta=0.00030  val_loss=-0.8004  final_kl=161.1599  final_recon=-0.8488  ok  (88s)
  beta=0.00010  val_loss=-0.7252  final_kl=218.8377  final_recon=-0.7470  ok  (88s)
  beta=0.00003  val_loss=-0.6962  final_kl=289.7876  final_recon=-0.7049  ok  (98s)

Largest non-collapsing beta_max = 0.1


## Phase B — Latent-dimension ablation at the fixed beta

In [5]:
latent_results = {}
for latent_dim in LATENT_GRID:
    t0 = time.time()
    _, best_val, final_kl, final_recon, n_epochs = train_reconprob_vae(latent_dim, BEST_BETA, SCREEN_EPOCHS, SCREEN_PATIENCE)
    elapsed = time.time() - t0
    collapsed = final_kl < COLLAPSE_KL_THRESH
    latent_results[latent_dim] = {'best_val': best_val, 'final_kl': final_kl, 'collapsed': collapsed}
    print(f'  latent_dim={latent_dim:3d}  val_loss={best_val:.4f}  final_kl={final_kl:.4f}  {"COLLAPSED" if collapsed else "ok"}  ({elapsed:.0f}s)')

valid_latents = {k: v for k, v in latent_results.items() if not v['collapsed']}
BEST_LATENT = min(valid_latents, key=lambda k: valid_latents[k]['best_val'])
print(f'\nBest latent_dim at beta={BEST_BETA}: {BEST_LATENT}  (val_loss={valid_latents[BEST_LATENT]["best_val"]:.4f})')

  latent_dim= 16  val_loss=0.9212  final_kl=1.2443  ok  (97s)
  latent_dim= 20  val_loss=1.3652  final_kl=2.4127  ok  (90s)
  latent_dim= 24  val_loss=2.1134  final_kl=7.5881  ok  (90s)
  latent_dim= 28  val_loss=5.6706  final_kl=36.7971  ok  (32s)
  latent_dim= 32  val_loss=1.3199  final_kl=1.5098  ok  (96s)
  latent_dim= 40  val_loss=1.3323  final_kl=1.4560  ok  (103s)

Best latent_dim at beta=0.1: 16  (val_loss=0.9212)


## Phase C — Full training of the chosen configuration

In [6]:
print(f'Full training: latent_dim={BEST_LATENT}, beta_max={BEST_BETA} ...')
t0 = time.time()
tuned_model, tuned_val_loss, tuned_final_kl, tuned_final_recon, tuned_epochs = train_reconprob_vae(
    BEST_LATENT, BEST_BETA, FULL_MAX_EPOCHS, FULL_PATIENCE)
elapsed = time.time() - t0
print(f'Done in {elapsed/60:.1f} min.  epochs={tuned_epochs}  val_loss={tuned_val_loss:.4f}  final_kl={tuned_final_kl:.4f}')
print(f'({"WARNING: looks collapsed" if tuned_final_kl < COLLAPSE_KL_THRESH else "non-trivial, ok"})')

Full training: latent_dim=16, beta_max=0.1 ...
Done in 7.7 min.  epochs=201  val_loss=0.6984  final_kl=3.6122
(non-trivial, ok)


## Step 3 — Full evaluation, identical protocol to `vae_eval.ipynb`

In [7]:
with torch.no_grad():
    mse_train = tuned_model.anomaly_score(X_train_t).numpy()
    mse_val = tuned_model.anomaly_score(X_val_t).numpy()
mu_train, sigma_train = float(mse_train.mean()), float(mse_train.std())
val_p99 = float(np.percentile(mse_val, 99))
print(f'Tuned model: mu_train={mu_train:.5f}  sigma_train={sigma_train:.5f}  val_p99 threshold={val_p99:.5f}')

def evaluate(scores, y_true, threshold):
    pred = (scores > threshold).astype(int)
    p, r, _ = precision_recall_curve(y_true, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    oracle_f1 = float(f1s[np.argmax(f1s)])
    return {
        'auc_roc': roc_auc_score(y_true, scores), 'auc_pr': average_precision_score(y_true, scores),
        'precision': precision_score(y_true, pred, zero_division=0), 'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0), 'oracle_f1': oracle_f1,
    }

tuned_results = {}
mse_by_set = {}
for name in ALL_SETS:
    X_t = torch.from_numpy(data[name])
    with torch.no_grad():
        mse = tuned_model.anomaly_score(X_t).numpy()
    mse_by_set[name] = mse
    tuned_results[name] = evaluate(mse, labels[name], val_p99)

deployed_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))

print(f'\n{"set":12s} {"model":14s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s} {"OracleF1":>9s}')
for name in ALL_SETS:
    t = tuned_results[name]
    d_auc = deployed_eval['auc'][name]
    d_pr = deployed_eval['precision_recall'][name]['val_p99']
    d_oracle = deployed_eval['oracle_ceiling'][name]['f1']
    print(f'{name:12s} {"recon_prob":14s} {t["auc_pr"]:8.4f} {t["auc_roc"]:9.4f} {t["f1"]:7.3f} {t["precision"]:10.3f} {t["recall"]:8.3f} {t["oracle_f1"]:9.4f}')
    print(f'{name:12s} {"deployed":14s} {d_auc["auc_pr"]:8.4f} {d_auc["auc_roc"]:9.4f} {d_pr["f1"]:7.3f} {d_pr["precision"]:10.3f} {d_pr["recall"]:8.3f} {d_oracle:9.4f}')
    print(f'  -> PR-AUC change: {t["auc_pr"]-d_auc["auc_pr"]:+.4f}   F1 change: {t["f1"]-d_pr["f1"]:+.3f}   Oracle-F1 change: {t["oracle_f1"]-d_oracle:+.4f}\n')

Tuned model: mu_train=0.08425  sigma_train=0.83535  val_p99 threshold=2.05844

set          model            PR-AUC   ROC-AUC      F1  Precision   Recall  OracleF1
cc1_test     recon_prob       0.1608    0.8325   0.114      0.071    0.297    0.1993
cc1_test     deployed         0.6014    0.8763   0.618      0.630    0.605    0.6857
  -> PR-AUC change: -0.4406   F1 change: -0.504   Oracle-F1 change: -0.4864

drift_cc2    recon_prob       0.0151    0.6807   0.034      0.018    0.504    0.0357
drift_cc2    deployed         0.4089    0.8812   0.276      0.168    0.772    0.5147
  -> PR-AUC change: -0.3938   F1 change: -0.241   Oracle-F1 change: -0.4790



## Step 4 — Per-fault-type recall, reconstruction-probability vs. deployed

In [8]:
ft = {name: np.load(os.path.join(DATA_DIR, f'ft_{name}.npy'), allow_pickle=True) for name in ALL_SETS}
deployed_fault = deployed_eval['per_fault_recall']

print(f'{"set":12s} {"fault_type":14s} {"n":>5s} {"deployed recall":>16s} {"recon_prob recall":>18s}')
for name in ALL_SETS:
    pred = (mse_by_set[name] > val_p99).astype(int)
    types_present = sorted({v for v in ft[name] if isinstance(v, str)})
    for ftype in types_present:
        mask = ft[name] == ftype
        n = int(mask.sum())
        rec_tuned = pred[mask].mean() if n > 0 else float('nan')
        rec_deployed = deployed_fault[name][ftype]['recall']
        print(f'{name:12s} {ftype:14s} {n:5d} {rec_deployed:16.3f} {rec_tuned:18.3f}')
    print()

set          fault_type         n  deployed recall  recon_prob recall
cc1_test     cpu               88            0.580              0.261
cc1_test     memory            76            0.763              0.658
cc1_test     pod-failure       92            0.500              0.033

drift_cc2    cpu              207            0.908              0.667
drift_cc2    memory           414            0.713              0.483
drift_cc2    pod-failure       99            0.737              0.253



## Step 5 — Save (does NOT overwrite the deployed model)

In [9]:
save_results = {
    'beta_search': beta_search,
    'best_beta': BEST_BETA,
    'latent_search': latent_results,
    'best_config': {'latent_dim': BEST_LATENT, 'beta_max': BEST_BETA},
    'tuned_model_meta': {
        'input_dim': INPUT_DIM, 'hidden1': HIDDEN1, 'hidden2': HIDDEN2, 'latent_dim': BEST_LATENT,
        'beta_max': BEST_BETA, 'clip': CLIP, 'mu_train': mu_train, 'sigma_train': sigma_train,
        'val_p99': val_p99, 'epochs_trained': tuned_epochs,
    },
    'tuned_results': tuned_results,
    'deployed_comparison': {
        name: {'auc': deployed_eval['auc'][name], 'precision_recall': deployed_eval['precision_recall'][name]['val_p99'],
               'oracle_f1': deployed_eval['oracle_ceiling'][name]['f1']}
        for name in ALL_SETS
    },
}
out_path = os.path.join(OUT_DIR, 'reconstruction_probability_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

model_path = os.path.join(OUT_DIR, 'vae_cc1_reconprob.pt')
torch.save(tuned_model.state_dict(), model_path)
print(f'Model weights saved -> {model_path}  (NOT deployed - models/vae_cc1.pt is unchanged)')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\reconstruction_probability_results.pkl
Model weights saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\vae_cc1_reconprob.pt  (NOT deployed - models/vae_cc1.pt is unchanged)


## How to read this

- This is architecturally distinct from `train_vae.ipynb`'s model (two-headed
  decoder, Gaussian NLL scoring) — not a hyperparameter variant of it.
- Beta and latent_dim were re-searched from scratch for this new loss using
  the same valid two-phase methodology as `hyperparameter_tuning_v2.ipynb`
  (collapse-check for beta, then loss comparison only within that fixed beta).
- **Step 3 is the honest verdict.** If this beats the deployed model's F1/
  PR-AUC on `cc1_test` and/or `drift_cc2`, that is a genuine, mechanistically-
  motivated improvement worth reporting. If it doesn't, that is also a real,
  informative result — it would mean per-dimension uncertainty weighting
  isn't the missing piece for this specific dataset, narrowing down what
  actually would help.
- Nothing here overwrites `models/vae_cc1.pt`.